# Spin-1 XY numerical evidence for the current ICQMBS draft

This notebook is keyed to **Sec. VII and the draft numerical note around Fig. 5** of the July 20, 2026 draft. It deliberately does **not** repeat the older numerical program based on support entropy, generic cage census, or an undecomposed Fock-space graph.

The analytical section already establishes:

- the one-dimensional Type-I boundary kernel for the $\pi$-bimagnon tower;
- the bounded two-site reduced-IZ witness $Z^{\mathrm{red}}_{r,r+1}$ and one-site potential witness $Y_r$;
- exact finite-$L$ infinite-temperature activities in a fixed-magnetization sector;
- the bondwise phase-compatibility condition and the site-diagonal preserving family.

The numerical objectives are therefore narrower:

1. certify the finite-size boundary kernel and local annihilation residuals;
2. verify the exact finite-$L$ activity ratios and their asymptotic limits;
3. compare symmetry-resolved microcanonical second moments with those ratios;
4. establish a thermal spectral background after full desymmetrization;
5. verify exactness and thermal activity for a phase-compatible deformation, plus one nontrivial inhomogeneous-$D_r$ case;
6. show how the scar residual opens when the phase condition is weakly violated.

All plotting cells use plain Matplotlib defaults so the presentation style can be adjusted manually.

## Imports and reproducibility settings

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy.linalg as la

from qlinks.basis.configs import basis_configs_from_build_result
from qlinks.caging import (
    LocalWitnessTemplate,
    adjacent_gap_ratio_report,
    basis_permutation_from_variable_permutation,
    cyclic_symmetry_sector_basis,
    diagnose_cage_stability,
    diagnose_eigenpair,
    eigenstate_expectations,
    evaluate_local_witness_on_diagonal_ensemble,
    evaluate_local_witness_on_states,
    permutation_matrix,
    project_operator_to_sector,
    project_state_to_sector,
    refine_sector_by_involution,
    select_microcanonical_window_by_count,
)
from qlinks.models import (
    SpinOneXYChainModel,
    spin_one_xy_periodic_range_couplings,
    spin_one_xy_phase_compatibility,
    spin_one_xy_scar_tower_states,
    spin_one_xy_tower_thermal_activities,
)
from helpers import set_revtex_matplotlib_style

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parents[1]
DATA_DIR = ROOT / "experimental" / "data" / "spin1_xy_draft_evidence"
DATA_DIR.mkdir(parents=True, exist_ok=True)

TOL = 1.0e-10
SIZES = (6, 8, 10, 12)
TOTAL_SZ = -2
J_DRAFT = 1.0
J1_MATRIX = 2.0 * J_DRAFT  # qlinks matrix element; draft Eq. (113) is 2J
J3_MATRIX = 0.74
D_THERMAL = 0.63
WINDOW_FRACTION = 0.20

print("data directory:", DATA_DIR)
print("sizes:", SIZES, "fixed total Sz:", TOTAL_SZ)
print("draft J:", J_DRAFT, "qlinks nearest-neighbor matrix element:", J1_MATRIX)

set_revtex_matplotlib_style(base_font_size=16)

## Local witnesses and symmetry-sector helpers

In [ ]:
def make_spin1_witnesses(*, xy_matrix_element: float = J1_MATRIX):
    # Y_r = (Sz_r)^2 - 1.  The thermodynamic observable is Y_r^2 = |0><0|.
    y_template = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=((0,),),
        local_operator=np.asarray([[-1.0]], dtype=np.complex128),
        metadata={"name": "Y_r", "support_sites": 1},
    )
    y = y_template.instantiate((0,))

    # Zred = c[|00>(<+ -| + <- +|) + h.c.], c = 2J in the draft convention.
    z_operator = np.asarray(
        [
            [0.0, xy_matrix_element, xy_matrix_element],
            [xy_matrix_element, 0.0, 0.0],
            [xy_matrix_element, 0.0, 0.0],
        ],
        dtype=np.complex128,
    )
    z_template = LocalWitnessTemplate(
        pattern_key=(),
        local_patterns=((0, 0), (1, -1), (-1, 1)),
        local_operator=z_operator,
        metadata={"name": "Zred_r_r+1", "support_sites": 2},
    )
    z = z_template.instantiate((0, 1))
    return y, z


def tower_state_for_sector(basis_configs: np.ndarray, *, length: int) -> np.ndarray:
    states, labels = spin_one_xy_scar_tower_states(
        basis_configs=basis_configs,
        length=length,
        normalize=True,
    )
    if states.shape[1] != 1:
        raise RuntimeError(f"expected one tower state in a fixed-M basis, found {labels}")
    return states[:, 0]


def tower_symmetry_sector(basis_configs: np.ndarray, scar: np.ndarray, *, length: int):
    n_raised = (TOTAL_SZ + length) // 2
    momentum_index = 0 if n_raised % 2 == 0 else length // 2

    translation = basis_permutation_from_variable_permutation(
        basis_configs,
        np.roll(np.arange(length), 1),
    )
    sector = cyclic_symmetry_sector_basis(
        translation,
        order=length,
        momentum_index=momentum_index,
        labels={"total_sz": TOTAL_SZ},
    )

    # Reflection r -> -r.  k=0 and k=pi sectors are invariant under reflection.
    reflection = basis_permutation_from_variable_permutation(
        basis_configs,
        (-np.arange(length)) % length,
    )
    reflection_value = complex(np.vdot(scar, permutation_matrix(reflection) @ scar))
    reflection_parity = 1 if reflection_value.real >= 0.0 else -1
    sector = refine_sector_by_involution(
        sector,
        reflection,
        eigenvalue=reflection_parity,
        label="reflection",
    )
    return sector, momentum_index, reflection_parity


def projected_witness_square(witness, basis_configs, sector):
    local_operator = witness.embed(basis_configs)
    q_operator = local_operator.conj().T @ local_operator
    return project_operator_to_sector(q_operator, sector)


def periodic_phase_compatible_model(*, length: int, d_z: float):
    return SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=J1_MATRIX,
        d_z=d_z,
        total_sz=TOTAL_SZ,
        extra_xy_couplings=spin_one_xy_periodic_range_couplings(
            length=length,
            distance=3,
            coefficient=J3_MATRIX,
        ),
    )

Y_WITNESS, Z_WITNESS = make_spin1_witnesses()
print("||Y|| =", Y_WITNESS.template.operator_norm)
print("||Zred|| =", Z_WITNESS.template.operator_norm)
print("||Zred^2|| =", Z_WITNESS.template.q_operator_norm)

## A. Exact Type-I boundary kernel and bounded local darkness

In [ ]:
L_REP = 8
model_rep = periodic_phase_compatible_model(length=L_REP, d_z=0.0)
build_rep = model_rep.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
    on_missing="raise",
)
configs_rep = basis_configs_from_build_result(build_rep)
scar_rep = tower_state_for_sector(configs_rep, length=L_REP)
support_rep = np.flatnonzero(np.abs(scar_rep) > TOL)

stability_rep = diagnose_cage_stability(
    build_rep.kinetic,
    support_rep,
    state=scar_rep,
    tolerance=TOL,
)
eigenpair_rep = diagnose_eigenpair(build_rep.hamiltonian, scar_rep)
y_rep = evaluate_local_witness_on_states(
    Y_WITNESS,
    basis_configs=configs_rep,
    states=scar_rep,
)
z_rep = evaluate_local_witness_on_states(
    Z_WITNESS,
    basis_configs=configs_rep,
    states=scar_rep,
)

local_rows = []
for descriptor in model_rep.local_term_descriptors(operator_kind="kinetic", term_kind="bond"):
    local_matrix = model_rep.build_local_term(descriptor, build_rep, builder="optimized")
    local_rows.append(
        {
            "term": descriptor.label,
            "sites": descriptor.support_sites,
            "action_norm": float(np.linalg.norm(local_matrix @ scar_rep)),
        }
    )
local_term_df = pd.DataFrame(local_rows)

boundary_scorecard = pd.DataFrame(
    [
        {
            "L": L_REP,
            "M": TOTAL_SZ,
            "full_sector_dimension": configs_rep.shape[0],
            "support_size": support_rep.size,
            "boundary_rank": stability_rep.boundary_rank,
            "boundary_nullity": stability_rep.boundary_nullity,
            "boundary_singular_gap": stability_rep.interference_gap,
            "boundary_residual": stability_rep.state_boundary_residual,
            "internal_residual": stability_rep.state_internal_eigen_residual,
            "full_eigenpair_residual": eigenpair_rep.residual_norm,
            "Y_annihilation_residual": y_rep.annihilation_residual,
            "Z_annihilation_residual": z_rep.annihilation_residual,
        }
    ]
)

display(boundary_scorecard)
display(local_term_df)
boundary_scorecard.to_csv(DATA_DIR / "boundary_kernel_scorecard.csv", index=False)
local_term_df.to_csv(DATA_DIR / "local_term_annihilation.csv", index=False)

The representative calculation should show:

- shell support $\binom{L}{n}$;
- boundary nullity one;
- a nonzero smallest positive singular value of the boundary map;
- zero action of every phase-compatible odd-range exchange term;
- zero $Y_r^2$ and $(Z^{\rm red}_{r,r+1})^2$ expectation in the scar.

This is the finite-size numerical certificate of the analytical relations, not the thermal-background evidence.

## B. Exact finite-L thermal activities and analytical limits

In [ ]:
formula_rows = []
for length in range(4, 32, 2):
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    formula_rows.append(exact.to_summary_dict())
formula_df = pd.DataFrame(formula_rows)

# Independent direct traces in the qlinks fixed-M basis for the ED-accessible sizes.
direct_rows = []
for length in (4, 6, 8, 10):
    result = SpinOneXYChainModel(
        length=length,
        boundary_condition="periodic",
        j_xy=0.0,
        total_sz=TOTAL_SZ,
    ).build(builder="optimized", basis_solver="dfs", sort_basis=True)
    configs = basis_configs_from_build_result(result)
    y_eval = evaluate_local_witness_on_diagonal_ensemble(
        Y_WITNESS,
        basis_configs=configs,
    )
    z_eval = evaluate_local_witness_on_diagonal_ensemble(
        Z_WITNESS,
        basis_configs=configs,
    )
    direct_rows.append(
        {
            "length": length,
            "basis_dimension": configs.shape[0],
            "Y2_direct_trace": y_eval.expectation,
            "Z2_direct_trace": z_eval.expectation,
        }
    )
direct_df = pd.DataFrame(direct_rows)
activity_df = formula_df.merge(direct_df, how="left", on="length")
activity_df["Y2_direct_minus_formula"] = (
    activity_df["Y2_direct_trace"] - activity_df["y2_activity"]
)
activity_df["Z2_direct_minus_formula"] = (
    activity_df["Z2_direct_trace"] - activity_df["z2_activity"]
)

display(activity_df.head(8))
activity_df.to_csv(DATA_DIR / "exact_fixed_M_activities.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(activity_df["length"], activity_df["y2_activity"], marker="o", label=r"exact finite $L$")
ax.plot(activity_df["length"], activity_df["y2_limit"], marker=".", linestyle="--", label=r"$p_0(q_L)$")
ax.axhline(1.0 / 3.0, linestyle=":", label=r"$q \to 0$ limit")
ax.set_xlabel(r"$L$")
ax.set_ylabel(r"$\mathrm{Tr}(\rho_{\infty,M}Y_r^2)$")
ax.legend()
ax.grid()
fig.tight_layout()

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(activity_df["length"], activity_df["z2_activity"], marker="o", label=r"exact finite $L$")
ax.plot(activity_df["length"], activity_df["z2_limit"], marker=".", linestyle="--", label=r"$16 J^2 p_0(q_L)^2$")
ax.axhline(4.0 * J1_MATRIX**2 / 9.0, linestyle=":", label=r"$q \to 0$ limit")
ax.set_xlabel(r"$L$")
ax.set_ylabel(r"$\mathrm{Tr}[\rho_{\infty,M}(Z^{\rm red})^2]$")
ax.legend()
ax.grid()
fig.tight_layout()

## C. Symmetry-resolved microcanonical activities, ETH scatter, and level statistics

In [ ]:
spectral_rows = []
scan_cache = {}

for length in SIZES:
    t0 = time.perf_counter()
    n_raised = (TOTAL_SZ + length) // 2

    result_zero = periodic_phase_compatible_model(length=length, d_z=0.0).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    configs = basis_configs_from_build_result(result_zero)
    scar = tower_state_for_sector(configs, length=length)
    sector, momentum_index, reflection_parity = tower_symmetry_sector(
        configs,
        scar,
        length=length,
    )
    scar_sector = project_state_to_sector(scar, sector)
    qy_sector = projected_witness_square(Y_WITNESS, configs, sector)
    qz_sector = projected_witness_square(Z_WITNESS, configs, sector)

    h0_sector = project_operator_to_sector(result_zero.hamiltonian, sector)
    e0, v0 = la.eigh(h0_sector)
    y0 = eigenstate_expectations(qy_sector, v0)
    z0 = eigenstate_expectations(qz_sector, v0)
    target_count = min(
        sector.sector_dimension,
        max(10, int(math.ceil(WINDOW_FRACTION * sector.sector_dimension))),
    )
    window0 = select_microcanonical_window_by_count(
        e0,
        target_energy=0.0,
        target_count=target_count,
        include_boundary_degeneracy=True,
    )
    idx0 = np.asarray(window0.indices, dtype=np.int64)

    result_d = periodic_phase_compatible_model(length=length, d_z=D_THERMAL).build(
        builder="optimized",
        basis_solver="dfs",
        sort_basis=True,
    )
    np.testing.assert_array_equal(result_d.basis.states, result_zero.basis.states)
    hd_sector = project_operator_to_sector(result_d.hamiltonian, sector)
    ed, vd = la.eigh(hd_sector)
    yd = eigenstate_expectations(qy_sector, vd)
    zd = eigenstate_expectations(qz_sector, vd)
    scar_energy = D_THERMAL * length
    overlap = np.abs(vd.conj().T @ scar_sector)
    scar_level = int(np.argmax(overlap))
    windowd = select_microcanonical_window_by_count(
        ed,
        target_energy=scar_energy,
        target_count=target_count,
        include_boundary_degeneracy=True,
    )
    idxd = np.asarray(windowd.indices, dtype=np.int64)
    gap = adjacent_gap_ratio_report(
        ed,
        trim_fraction=0.10,
        degeneracy_tolerance=1.0e-8,
    )
    exact = spin_one_xy_tower_thermal_activities(
        length=length,
        total_sz=TOTAL_SZ,
        xy_matrix_element=J1_MATRIX,
    )
    residual_d = diagnose_eigenpair(result_d.hamiltonian, scar)

    spectral_rows.append(
        {
            "L": length,
            "M": TOTAL_SZ,
            "n_raised": n_raised,
            "full_M_sector_dimension": configs.shape[0],
            "momentum_index": momentum_index,
            "momentum_over_pi": 2.0 * momentum_index / length,
            "reflection_parity": reflection_parity,
            "resolved_sector_dimension": sector.sector_dimension,
            "D": D_THERMAL,
            "scar_energy": scar_energy,
            "scar_level_energy": ed[scar_level],
            "scar_overlap": overlap[scar_level],
            "scar_residual": residual_d.residual_norm,
            "D0_window_half_width": window0.half_width,
            "D0_window_state_count": window0.n_states,
            "D0_window_center_offset": window0.center_offset,
            "D0_microcanonical_Y2": float(np.mean(y0[idx0])),
            "D0_microcanonical_Z2": float(np.mean(z0[idx0])),
            "exact_fixed_M_Y2": exact.y2_activity,
            "exact_fixed_M_Z2": exact.z2_activity,
            "finiteD_window_half_width": windowd.half_width,
            "finiteD_window_state_count": windowd.n_states,
            "finiteD_window_center_offset": windowd.center_offset,
            "finiteD_microcanonical_Y2": float(np.mean(yd[idxd])),
            "finiteD_microcanonical_Z2": float(np.mean(zd[idxd])),
            "mean_gap_ratio": gap.mean_ratio,
            "gap_ratio_count": len(gap.ratios),
            "runtime_seconds": time.perf_counter() - t0,
        }
    )
    scan_cache[length] = {
        "configs": configs,
        "scar": scar,
        "sector": sector,
        "energies_D": ed,
        "vectors_D": vd,
        "Y2_D": yd,
        "Z2_D": zd,
        "scar_level": scar_level,
        "window_D": windowd,
        "gap_report": gap,
    }

spectral_df = pd.DataFrame(spectral_rows)
display(spectral_df)
spectral_df.to_csv(DATA_DIR / "symmetry_resolved_spectral_evidence.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(
    spectral_df["L"],
    spectral_df["D0_microcanonical_Y2"] / spectral_df["exact_fixed_M_Y2"],
    marker="o",
    label=r"$Y_r^2$",
)
ax.plot(
    spectral_df["L"],
    spectral_df["D0_microcanonical_Z2"] / spectral_df["exact_fixed_M_Z2"],
    marker="s",
    label=r"$(Z^{\rm red})^2$",
)
ax.axhline(1.0, linestyle=":")
ax.set_xlabel("L")
ax.set_ylabel("microcanonical / exact fixed-M activity")
ax.legend()
fig.tight_layout()

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(spectral_df["L"], spectral_df["mean_gap_ratio"], marker="o", label="resolved data")
ax.axhline(0.5307, linestyle="--", label="GOE")
ax.axhline(2.0 * np.log(2.0) - 1.0, linestyle=":", label="Poisson")
ax.set_xlabel("L")
ax.set_ylabel(r"mean adjacent-gap ratio $\langle r\rangle$")
ax.legend()
fig.tight_layout()

The $D=0$ microcanonical comparison must be interpreted with care: chiral symmetry produces a finite zero-mode manifold in the resolved sector. The window selection includes complete degeneracies at its boundary. The finite-$D$ data lift this accidental zero-mode degeneracy while leaving the tower exact, and are therefore the cleaner level-statistics benchmark.

### ETH scatter at the largest resolved size

In [ ]:
largest = scan_cache[max(SIZES)]
energies = largest["energies_D"]
y_values = largest["Y2_D"]
z_values = largest["Z2_D"]
scar_level = largest["scar_level"]

scatter_df = pd.DataFrame(
    {
        "energy": energies,
        "energy_density": energies / max(SIZES),
        "Y2": y_values,
        "Z2": z_values,
        "is_scar_level": np.arange(energies.size) == scar_level,
    }
)
scatter_df.to_csv(DATA_DIR / "eth_scatter_Lmax_finite_D.csv", index=False)

fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.scatter(scatter_df["energy_density"], scatter_df["Y2"], s=12, alpha=0.65, label=r"$Y_r^2$")
ax.scatter(scatter_df["energy_density"], scatter_df["Z2"], s=12, alpha=0.65, label=r"$(Z^{\rm red})^2$")
ax.scatter(
    [scatter_df.loc[scar_level, "energy_density"]],
    [0.0],
    marker="*",
    s=150,
    label=r"exact tower state",
)
ax.set_xlabel(r"energy density")
ax.set_ylabel(r"local second moment")
ax.legend()
ax.grid()
fig.tight_layout()

display(scatter_df.iloc[max(0, scar_level - 3): scar_level + 4])

In [ ]:
fig.savefig(
    "../images/spin1_xy_eth_scatter.pdf",
    bbox_inches="tight",
    pad_inches=0.02,
)

## D. Bondwise deformation condition and controlled violation

In [ ]:
L_DEF = 8
phases = (-1.0) ** np.arange(L_DEF)
nearest_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=1,
    coefficient=J1_MATRIX,
)
third_pairs = spin_one_xy_periodic_range_couplings(
    length=L_DEF,
    distance=3,
    coefficient=J3_MATRIX,
)
compatibility = spin_one_xy_phase_compatibility(
    nearest_pairs + third_pairs,
    phases=phases,
)
compatibility_df = pd.DataFrame(
    [
        {
            "site_i": pair[0],
            "site_j": pair[1],
            "coupling": coupling,
            "phase_condition_residual": residual,
            "absolute_residual": abs(residual),
        }
        for pair, coupling, residual in zip(
            compatibility.pairs,
            compatibility.couplings,
            compatibility.residuals,
            strict=True,
        )
    ]
)
display(compatibility_df)
compatibility_df.to_csv(DATA_DIR / "bondwise_phase_compatibility.csv", index=False)

base_result = periodic_phase_compatible_model(length=L_DEF, d_z=D_THERMAL).build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
base_configs = basis_configs_from_build_result(base_result)
base_scar = tower_state_for_sector(base_configs, length=L_DEF)
violating_result = SpinOneXYChainModel(
    length=L_DEF,
    boundary_condition="periodic",
    j_xy=0.0,
    total_sz=TOTAL_SZ,
    extra_xy_couplings=((0, 2, 1.0),),  # same-sublattice exchange violates Eq. (134)
).build(builder="optimized", basis_solver="dfs", sort_basis=True)
np.testing.assert_array_equal(violating_result.basis.states, base_result.basis.states)

violation_rows = []
for epsilon in np.linspace(0.0, 0.20, 11):
    hamiltonian = base_result.hamiltonian + epsilon * violating_result.hamiltonian
    report = diagnose_eigenpair(hamiltonian, base_scar)
    phase_report = spin_one_xy_phase_compatibility(
        nearest_pairs + third_pairs + ((0, 2, epsilon),),
        phases=phases,
    )
    violation_rows.append(
        {
            "epsilon": epsilon,
            "max_phase_condition_residual": phase_report.max_residual,
            "scar_residual": report.residual_norm,
            "scar_variance": report.variance,
        }
    )
violation_df = pd.DataFrame(violation_rows)
display(violation_df)
violation_df.to_csv(DATA_DIR / "phase_condition_violation.csv", index=False)

fig, ax = plt.subplots(figsize=(6.4, 4.0))
ax.plot(violation_df["epsilon"], violation_df["scar_residual"], marker="o")
ax.set_xlabel(r"phase-incompatible coupling $\epsilon$")
ax.set_ylabel(r"$\|(H-E)|S_n\rangle\|$")
ax.grid()
fig.tight_layout()

## E. Inhomogeneous single-ion anisotropy with a phase-compatible bond-disordered thermal background

In [ ]:
L_INHOM = 8
rng = np.random.default_rng(13)
sites = np.arange(L_INHOM)

# Arbitrary real exchanges between opposite sublattices satisfy Eq. (134) for eta_r=(-1)^r.
# Random bond strengths break translation and reflection while preserving the tower exactly.
inhom_pairs = []
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=1,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(1.5 + 0.8 * rng.random())))
for site_i, site_j, _ in spin_one_xy_periodic_range_couplings(
    length=L_INHOM,
    distance=3,
    coefficient=1.0,
):
    inhom_pairs.append((site_i, site_j, float(0.2 + 0.8 * rng.random())))

d_profile = 0.4 + 0.4 * rng.random(L_INHOM)
inhom_phase = spin_one_xy_phase_compatibility(
    tuple(inhom_pairs),
    phases=(-1.0) ** sites,
)
assert inhom_phase.is_compatible

inhom_model = SpinOneXYChainModel(
    length=L_INHOM,
    boundary_condition="periodic",
    j_xy=0.0,
    d_z_by_site=tuple(float(value) for value in d_profile),
    total_sz=TOTAL_SZ,
    extra_xy_couplings=tuple(inhom_pairs),
)
inhom_result = inhom_model.build(
    builder="optimized",
    basis_solver="dfs",
    sort_basis=True,
)
inhom_configs = basis_configs_from_build_result(inhom_result)
inhom_scar = tower_state_for_sector(inhom_configs, length=L_INHOM)
inhom_residual = diagnose_eigenpair(inhom_result.hamiltonian, inhom_scar)
inhom_scar_energy = float(np.sum(d_profile))

# Spatial symmetries are deliberately broken, so the fixed-M block is already desymmetrized.
inhom_h = inhom_result.hamiltonian.toarray()
inhom_energies, inhom_vectors = la.eigh(inhom_h)
y_local = Y_WITNESS.embed(inhom_configs)
z_local = Z_WITNESS.embed(inhom_configs)
y2_inhom = eigenstate_expectations(y_local.conj().T @ y_local, inhom_vectors)
z2_inhom = eigenstate_expectations(z_local.conj().T @ z_local, inhom_vectors)
inhom_overlap = np.abs(inhom_vectors.conj().T @ inhom_scar)
inhom_scar_level = int(np.argmax(inhom_overlap))
inhom_window = select_microcanonical_window_by_count(
    inhom_energies,
    target_energy=inhom_scar_energy,
    target_count=80,
    include_boundary_degeneracy=True,
)
inhom_indices = np.asarray(inhom_window.indices, dtype=np.int64)
inhom_gap = adjacent_gap_ratio_report(
    inhom_energies,
    trim_fraction=0.10,
    degeneracy_tolerance=1.0e-8,
)

inhom_df = pd.DataFrame(
    [
        {
            "L": L_INHOM,
            "M": TOTAL_SZ,
            "full_sector_dimension": inhom_configs.shape[0],
            "max_phase_condition_residual": inhom_phase.max_residual,
            "scar_energy_expected": inhom_scar_energy,
            "scar_energy_eigensolver": inhom_energies[inhom_scar_level],
            "scar_overlap": inhom_overlap[inhom_scar_level],
            "scar_residual": inhom_residual.residual_norm,
            "window_half_width": inhom_window.half_width,
            "window_state_count": inhom_window.n_states,
            "window_center_offset": inhom_window.center_offset,
            "microcanonical_Y2": float(np.mean(y2_inhom[inhom_indices])),
            "microcanonical_Z2": float(np.mean(z2_inhom[inhom_indices])),
            "mean_gap_ratio": inhom_gap.mean_ratio,
            "gap_ratio_count": len(inhom_gap.ratios),
        }
    ]
)
inhom_profile_df = pd.DataFrame({"site": sites, "D_r": d_profile})
inhom_coupling_df = pd.DataFrame(
    [
        {
            "site_i": site_i,
            "site_j": site_j,
            "matrix_element": coupling,
        }
        for site_i, site_j, coupling in inhom_pairs
    ]
)
display(inhom_profile_df)
display(inhom_coupling_df)
display(inhom_df)
inhom_profile_df.to_csv(DATA_DIR / "inhomogeneous_D_profile.csv", index=False)
inhom_coupling_df.to_csv(DATA_DIR / "inhomogeneous_phase_compatible_couplings.csv", index=False)
inhom_df.to_csv(DATA_DIR / "inhomogeneous_D_evidence.csv", index=False)

## Data manifest

In [ ]:
manifest = pd.DataFrame(
    [
        {"file": path.name, "bytes": path.stat().st_size}
        for path in sorted(DATA_DIR.glob("*.csv"))
    ]
)
display(manifest)
manifest.to_csv(DATA_DIR / "manifest.csv", index=False)
print("All numerical tables were written to", DATA_DIR)

## Interpretation checklist for the manuscript

The numerical section should make the following distinctions explicit:

- **Analytical**: tower wavefunction, one-dimensional boundary kernel, local witness annihilation, exact fixed-$M$ activity ratios, and the preserving conditions.
- **Numerically corroborated**: finite-size residuals and direct traces.
- **Numerical evidence for the scar claim**: symmetry-resolved ETH scatter and GOE-like level statistics around the finite-$D$ tower energy.
- **Finite-$D$ / inhomogeneous extension**: exactness remains analytical, while the finite-temperature local activity is measured numerically.
- **Caution at $D=0$**: the chiral zero-mode manifold must not be cut by an arbitrary eigenbasis or an energy window that splits an exact degeneracy. The notebook includes complete boundary degeneracies and reports the window explicitly.

The old automorphism-colored Fock-space graph is not required for the main numerical figure and can remain an appendix orientation plot.